# E006 — Assemble & Validate V2 Submission

**Input checklist**
- Required: `learned_model.json` from E004.
- Recommended: `cem_best.json` from E005.
- Accelerator: **None / CPU**.
- Internet: **OFF**.

**Output:** `submission_v2.tar.gz`. The submission contains only the deterministic controller, tiny model runtime, learned JSON, and entrypoint.

In [ ]:
from pathlib import Path
import os,sys,json
SUITE_CANDIDATES=[Path('/kaggle/input/kaggriculture-v2-suite'),Path('/kaggle/input/kaggriculture-v2-suite/kaggriculture_v2_suite'),Path.cwd().parent,Path.cwd()]
ROOT=next((p for p in SUITE_CANDIDATES if (p/'src'/'kagv2').exists()),None)
if ROOT is None: raise FileNotFoundError('Attach/upload kaggriculture_v2_suite as a Kaggle Dataset, or run this notebook inside the repo.')
sys.path.insert(0,str(ROOT)); WORK=Path('/kaggle/working/kagv2') if Path('/kaggle/working').exists() else ROOT/'artifacts'; WORK.mkdir(parents=True,exist_ok=True)
print('ROOT=',ROOT,'WORK=',WORK)

In [ ]:
import json,shutil,sys,subprocess,tarfile,py_compile
SUB=ROOT/'submission'
shutil.copy2(WORK/'learned_model.json',SUB/'learned_model.json')
model=json.loads((SUB/'learned_model.json').read_text())
if (WORK/'cem_best.json').exists():
    best=json.loads((WORK/'cem_best.json').read_text())['best'];n=len((model.get('archetype') or {}).get('centroids',[]));model['policy_by_archetype']={str(i):best for i in range(n)}
    (SUB/'learned_model.json').write_text(json.dumps(model,indent=2,sort_keys=True))
for f in ['main.py','predictive_agent.py','parametric_agent.py','base_controller.py','runtime_model.py']:
    py_compile.compile(str(SUB/f),doraise=True)
print('model bytes',(SUB/'learned_model.json').stat().st_size)

In [ ]:
sys.path.insert(0,str(SUB));from predictive_agent import PredictiveMind
from src.kagv2.simulator import Game
from submission.base_controller import HarvestMind
for seed in range(3):
    s=Game(seed=seed).run([PredictiveMind().act,HarvestMind().act]);print(seed,s)

In [ ]:
OUT=WORK/'submission_v2.tar.gz';files=['main.py','predictive_agent.py','parametric_agent.py','base_controller.py','runtime_model.py','learned_model.json']
with tarfile.open(OUT,'w:gz') as t:
    for f in files:t.add(SUB/f,arcname=f)
print('READY:',OUT,'bytes=',OUT.stat().st_size)
with tarfile.open(OUT) as t:print(t.getnames())

### Before spending a leaderboard submission
1. Run a both-seat tournament against V1 and the counter policy.
2. If `kaggle_environments` is available, run an official-engine parity smoke test.
3. Verify inference stays far below the 1-second per-turn limit.
4. Submit one promoted V2, then mine its hosted episodes before using more daily slots.